# 05 — Phenology Overlap (Small-Scale Testing)

Computes the temporal overlap coefficient Δ for each of the 18 clean
interaction edges across all spatially co-occurring bins.

**Method:** For each (plant, pollinator, bin) triple:
1. Build normalized 52-week flowering histogram for the plant (from PhenoField)
2. Build normalized 52-week activity histogram for the pollinator (from GBIF)
3. Δ = Σ_t min(f̃_t, ã_t) — coefficient of overlapping (Ridout & Linkie 2009)

Minimum observation threshold: 10 records per (species, bin) to compute a
stable weekly histogram.

**Key finding from this analysis:**
A substantial fraction of spatially co-occurring (edge × bin) combinations
show low temporal overlap (Δ < 0.3), confirming that spatial co-occurrence
alone overestimates interaction likelihood. This motivated the inclusion of
temporal signal in ANTHEIA.

**Output:** `overlap_coefficients.csv`, `spatial_vs_temporal.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, pearsonr
from pathlib import Path

BASE       = Path("/scratch/ariana.l")
PLANTS_IN  = BASE / "Plant Pollinator Initial Analysis" / "plant_flowering_events.parquet"
POLL_IN    = BASE / "Plant Pollinator Initial Analysis" / "pollinator_observations_v2.csv"
EDGES_IN   = BASE / "CfE2026CVforEcology" / "rawpollinatordata" / "plant_pollinator_edges.csv"
OUT_DIR    = BASE

BIN_SIZE   = 1.5
MIN_OBS    = 10
N_WEEKS    = 52

print("Paths OK")

In [ ]:
# Load data
plants = pd.read_parquet(PLANTS_IN)
pollinators = pd.read_csv(POLL_IN, low_memory=False)
edges = pd.read_csv(EDGES_IN)

# Add spatial bins
for df, lat_col, lon_col in [(plants, 'lat', 'lon'), (pollinators, 'lat', 'lon')]:
    df['lat_bin'] = (np.floor(df[lat_col] / BIN_SIZE) * BIN_SIZE).round(6)
    df['lon_bin'] = (np.floor(df[lon_col] / BIN_SIZE) * BIN_SIZE).round(6)

# Add week
plants['week'] = ((plants['doy'].astype(int) - 1) // 7).clip(0, 51)
pollinators['week'] = ((pollinators['doy'].astype(int) - 1) // 7).clip(0, 51)

print(f"Edges: {len(edges)}")
print(f"Plants: {len(plants):,} records")
print(f"Pollinators: {len(pollinators):,} records")

In [ ]:
def weekly_density(df, species_col, species, lat_bin, lon_bin):
    """Return normalized 52-week histogram or None if < MIN_OBS records."""
    subset = df[
        (df[species_col] == species) &
        (df['lat_bin'] == lat_bin) &
        (df['lon_bin'] == lon_bin)
    ]
    if len(subset) < MIN_OBS:
        return None
    counts = np.zeros(N_WEEKS)
    for w, c in subset['week'].value_counts().items():
        counts[int(w)] = c
    return counts / counts.sum()

# Compute overlap for every (edge, bin) combination
print("Computing temporal overlap coefficients...")
results = []
for _, edge in edges.iterrows():
    plant = edge['plant_species']
    pollinator = edge['pollinator_species']

    plant_bins = set(zip(
        plants[plants['species'] == plant]['lat_bin'],
        plants[plants['species'] == plant]['lon_bin']
    ))
    pol_bins = set(zip(
        pollinators[pollinators['pollinator_species'] == pollinator]['lat_bin'],
        pollinators[pollinators['pollinator_species'] == pollinator]['lon_bin']
    ))

    for (lat_b, lon_b) in plant_bins & pol_bins:
        p_density = weekly_density(plants, 'species', plant, lat_b, lon_b)
        q_density = weekly_density(pollinators, 'pollinator_species', pollinator, lat_b, lon_b)
        if p_density is None or q_density is None:
            continue
        overlap = np.minimum(p_density, q_density).sum()
        results.append({
            'plant': plant, 'pollinator': pollinator,
            'lat_bin': lat_b, 'lon_bin': lon_b, 'overlap': overlap
        })

df_results = pd.DataFrame(results)
print(f"Total (edge × bin) combinations with ≥{MIN_OBS} obs on both sides: {len(df_results)}")
print(f"Unique edges represented: {df_results[['plant','pollinator']].drop_duplicates().shape[0]} / {len(edges)}")
print(f"\nOverlap distribution:")
print(df_results['overlap'].describe())

In [ ]:
# Key finding: fraction of spatially co-occurring pairs with low temporal overlap
THRESHOLD = 0.3
total = len(df_results)
low_overlap = (df_results['overlap'] < THRESHOLD).sum()
pct = 100 * low_overlap / total

print(f"Pairs with temporal overlap < {THRESHOLD}: {low_overlap} / {total} ({pct:.1f}%)")
print()
print("Interpretation: these are pairs where species co-occur spatially but")
print("are phenologically misaligned. A spatial-only model would incorrectly")
print("predict high interaction probability for these pairs.")

In [ ]:
# Spatial vs temporal overlap comparison
# Jaccard (spatial) vs Δ (temporal) per edge
spatial_temporal = []
for plant, pollinator in df_results[['plant', 'pollinator']].drop_duplicates().values:
    p_bins = set(zip(
        plants[plants['species'] == plant]['lat_bin'],
        plants[plants['species'] == plant]['lon_bin']
    ))
    q_bins = set(zip(
        pollinators[pollinators['pollinator_species'] == pollinator]['lat_bin'],
        pollinators[pollinators['pollinator_species'] == pollinator]['lon_bin']
    ))
    intersection = len(p_bins & q_bins)
    union = len(p_bins | q_bins)
    jaccard = intersection / union if union > 0 else 0
    temporal = df_results[
        (df_results['plant'] == plant) &
        (df_results['pollinator'] == pollinator)
    ]['overlap'].mean()
    spatial_temporal.append({
        'plant': plant, 'pollinator': pollinator,
        'spatial_jaccard': jaccard, 'temporal_overlap': temporal,
        'n_shared_bins': intersection
    })

st_df = pd.DataFrame(spatial_temporal).sort_values('spatial_jaccard', ascending=False)
sp_corr, sp_p = spearmanr(st_df['spatial_jaccard'], st_df['temporal_overlap'])
print(f"Spearman ρ (spatial vs temporal): {sp_corr:.3f} (p={sp_p:.3f})")
print()
print(st_df[['plant','pollinator','spatial_jaccard','temporal_overlap']].to_string(index=False))

In [ ]:
# Save
df_results.to_csv(OUT_DIR / 'overlap_coefficients.csv', index=False)
st_df.to_csv(OUT_DIR / 'spatial_vs_temporal.csv', index=False)
print("Saved overlap_coefficients.csv and spatial_vs_temporal.csv")